# Cortex Small-LLM Router GRPO training (Workstream B Phase 4 — SKELETON, post-pivot)

**Status:** scaffold with explicit `TODO(conv-point)` markers at every Cortex-dependent integration site. **Do not run end-to-end yet** — the cells marked TODO will fail until Workstream A's Session 13 ships:

- `cortex.metacognition.MetacognitionState` — featurization source.
- `cortex.routing_policy.RoutingPolicy` — trainable interface (Phase A §6).
- `cortex.council.Council` — deliberation orchestrator that drives rollouts.
- `baselines.cortex_fixed_router.B3CortexFixedRouter` — generates the deterministic-router corpus.

**Architecture (post-pivot — replaces the MLP-head approach in commit `5489e55`):**
- Router is a small LLM: `unsloth/Qwen3-1.5B-Instruct` + LoRA rank 16. Roughly 3 GB on a100-large in 4-bit.
- Input: NL summary of `MetacognitionState` (~300 tokens).
- Output: structured JSON matching `cortex.schemas.RoutingAction` (~150 tokens).
- GRPO via the same Unsloth + TRL `GRPOTrainer` pipeline as the B1 notebook (one less code path).
- Reward: Phase-1-fixed `outer_reward` ∈ [-1, 1] composed with the token-budget penalty via `training.reward_shaping.shape_reward`.

**Why a small LLM (not an MLP)?** With A100 compute available, a small LLM gives better reasoning over complex MetacognitionState, is interpretable in the pitch demo (you can read what the router thinks), and reuses the exact training infrastructure as the B1 baseline.

## 1. Install dependencies

Same Unsloth + vLLM + TRL stack as the B1 notebook; smaller GPU footprint because the trainable router is 1.5B.

In [ ]:
%%capture
!pip install --upgrade pip
!pip install unsloth vllm
!pip install --upgrade --no-deps "trl>=0.14" peft accelerate bitsandbytes
!pip install pydantic openenv huggingface_hub matplotlib

## 2. Authenticate with Hugging Face

In [ ]:
import os

try:
    from google.colab import userdata

    HF_TOKEN = userdata.get("HF_TOKEN")
    os.environ["HF_TOKEN"] = HF_TOKEN
except Exception:
    from huggingface_hub import login

    login()
    HF_TOKEN = os.environ.get("HF_TOKEN", "")

assert HF_TOKEN, "HF_TOKEN required"

## 3. Clone CrisisWorldCortex (post-Cortex-Session-13 deploy)

**TODO(conv-point):** the target Space repo at convergence point will have `cortex/*` populated through Session 13 (subagents + lenses + brains + council + metacognition + routing_policy + B3). Until then, the Cortex-dependent imports below will fail.

In [ ]:
%%capture
!rm -rf /content/CrisisWorldCortex
!git clone https://huggingface.co/spaces/Angshuman28/CrisisWorldCortex /content/CrisisWorldCortex
%cd /content/CrisisWorldCortex
!pip install -e .

In [ ]:
import sys

sys.path.insert(0, "/content/CrisisWorldCortex")


# TODO(conv-point): uncomment when Cortex Session 13 lands.
# from cortex.schemas import MetacognitionState, RoutingAction, RouterStep
# from cortex.routing_policy import RoutingPolicy
# from cortex.council import Council
# from baselines.cortex_fixed_router import B3CortexFixedRouter
print("Phase-2 training utilities OK; Cortex imports gated until Session 13")

## 4. Load Qwen3-1.5B-Instruct (router) with LoRA

Small enough to fit alongside frozen 7B/8B brain LLMs on a100-large (80GB). LoRA rank 16 — tighter than the B1 notebook's 32 because the action space is structured JSON (small effective vocabulary).

In [ ]:
from unsloth import FastLanguageModel

ROUTER_MODEL = "unsloth/Qwen3-1.5B-Instruct-bnb-4bit"
MAX_SEQ_LEN = 2048

router_model, router_tokenizer = FastLanguageModel.from_pretrained(
    model_name=ROUTER_MODEL,
    max_seq_length=MAX_SEQ_LEN,
    load_in_4bit=True,
    fast_inference=True,
    max_lora_rank=16,
    gpu_memory_utilization=0.5,  # share GPU with frozen brain LLMs at conv-point
)

router_model = FastLanguageModel.get_peft_model(
    router_model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=32,
    use_gradient_checkpointing="unsloth",
    random_state=42,
)
print("Router (Qwen3-1.5B + LoRA r=16) loaded")

## 5. Featurization: MetacognitionState → NL prompt + RoutingAction schema

Replaces the MLP version's `(24,) np.float32` featurization. The router consumes a natural-language summary; output is structured JSON validated against `cortex.schemas.RoutingAction`.

**TODO(conv-point):** the schema below references `cortex.schemas.MetacognitionState` (Session 13).
Per Phase A `cortex/schemas.py` and `docs/CORTEX_ARCHITECTURE.md` §6, the NL summary covers the 11 documented fields + the phase string.

In [ ]:
ROUTER_SYSTEM_PROMPT = """You are the Cortex router for CrisisWorldCortex. You receive a metacognition
state summary describing the current deliberation phase and emit ONE routing action as JSON.

Allowed action kinds (per cortex/CLAUDE.md):
- call_subagent: invoke a brain's subagent. Required: brain (epidemiology|logistics|governance),
  subagent (world_modeler|planner|critic).
- request_challenge: cross-brain critique. Required: challenger_brain, target_brain.
- switch_phase: advance the phase machine. Required: new_phase (divergence|challenge|narrowing|convergence).
- preserve_dissent: tag a minority recommendation. Required: tag (string, max 80 chars).
- emit_outer_action: close the tick with a final action.
- stop_and_no_op: close the tick with a no-op.

Hard caps (binding): ≤2 deliberation rounds/tick, ≤1 cross-brain challenge/tick,
≤1 critic call per brain/tick, ≤6000 token budget per tick.

Output exactly one JSON object — no markdown fences, no prose around it."""

PHASE_NAMES = ("divergence", "challenge", "narrowing", "convergence")


def metacog_state_to_prompt(state) -> str:
    """Convert MetacognitionState → NL summary for the router.

    TODO(conv-point): change ``state`` typing to MetacognitionState
    (cortex.schemas) once Session 13 lands. Duck-typed for now.
    """
    return (
        f"Tick {getattr(state, 'tick', 0)}, round {getattr(state, 'round', 1)}, "
        f"phase={getattr(state, 'phase', 'divergence')}.\n"
        f"Inter-brain agreement: {getattr(state, 'inter_brain_agreement', 0.0):.2f}.\n"
        f"Average confidence: {getattr(state, 'average_confidence', 0.0):.2f}.\n"
        f"Average evidence support: {getattr(state, 'average_evidence_support', 0.0):.2f}.\n"
        f"Novelty yield (last round): {getattr(state, 'novelty_yield_last_round', 0.0):.2f}.\n"
        f"Collapse suspicion: {getattr(state, 'collapse_suspicion', 0.0):.2f}.\n"
        f"Budget remaining: {getattr(state, 'budget_remaining_frac', 1.0):.0%}.\n"
        f"Urgency: {getattr(state, 'urgency', 0.0):.2f}.\n"
        f"Preserved dissent count: {getattr(state, 'preserved_dissent_count', 0)}.\n"
        f"Cross-brain challenge used this tick: "
        f"{bool(getattr(state, 'challenge_used_this_tick', 0))}.\n\n"
        f"Choose the next routing action."
    )


def make_router_chat_prompt(state) -> str:
    return router_tokenizer.apply_chat_template(
        [
            {"role": "system", "content": ROUTER_SYSTEM_PROMPT},
            {"role": "user", "content": metacog_state_to_prompt(state)},
        ],
        tokenize=False,
        add_generation_prompt=True,
    )

## 6. Connect to the deployed CrisisWorld env

Same env client pattern as the B1 notebook.

In [ ]:
from CrisisWorldCortex.client import CrisisworldcortexEnv

ENV_URL = "https://angshuman28-crisisworldcortex.hf.space"
TASKS = ("outbreak_easy", "outbreak_medium", "outbreak_hard")
EPISODE_TICKS = 12


def make_env() -> CrisisworldcortexEnv:
    return CrisisworldcortexEnv(base_url=ENV_URL)


_test_env = make_env()
_obs = _test_env.reset(task_name="outbreak_easy", seed=0, max_ticks=EPISODE_TICKS)
print(f"Env OK. Initial tick={_obs.tick}, regions={[r.region for r in _obs.regions]}")

## 7. Build training-data prompt set from B3 deterministic-router trajectories

**TODO(conv-point):** B3CortexFixedRouter runs ~50 episodes. Each `RouterStep`'s `MetacognitionState` becomes a router-prompt; the GRPO completion is the router's emitted JSON.

Replaces the MLP version's action-vocab + B3-trajectory-to-tuple pipeline. Same RolloutBuffer, different content shape.

In [ ]:
from datasets import Dataset

TRAIN_EPISODES = 50


def collect_b3_router_prompts(num_episodes: int = TRAIN_EPISODES) -> Dataset:
    """Run B3 and convert each RouterStep into a (prompt, task, seed) row.

    TODO(conv-point): uncomment the body when B3CortexFixedRouter ships.
    """
    rows = {"prompt": [], "task": [], "seed": []}
    # TODO(conv-point):
    # b3 = B3CortexFixedRouter(env=make_env())
    # for ep in range(num_episodes):
    #     trajectory = b3.run_episode(task="outbreak_easy", seed=ep)
    #     for router_step in trajectory.router_steps:
    #         rows["prompt"].append(make_router_chat_prompt(router_step.metacognition_state))
    #         rows["task"].append("outbreak_easy")
    #         rows["seed"].append(ep)
    return (
        Dataset.from_dict(rows)
        if rows["prompt"]
        else Dataset.from_dict(
            {"prompt": ["placeholder until conv-point"], "task": ["outbreak_easy"], "seed": [0]}
        )
    )


train_dataset = collect_b3_router_prompts()
print(f"Dataset: {len(train_dataset)} rows (will be ~{TRAIN_EPISODES * 8} at conv-point)")

## 8. Reward function: full-episode rollout per (prompt, completion)

Uses Phase-1 `outer_reward` summed across the episode, with Phase-2 `shape_reward` token-budget penalty.

**TODO(conv-point):** the rollout loop calls `Council.step` with the trainable router policy. Until Session 13 lands, the loop is stubbed and returns 0.0 for every (prompt, completion).

In [ ]:
def cortex_router_reward(prompts, completions, task, seed, **_kwargs):
    """Reward = sum(per-tick obs.reward) over a full episode driven by the
    trainable router's emitted RoutingAction JSON.

    Each (prompt, completion) pair represents ONE router decision; the
    full episode reward is shared across all router decisions in that
    episode (GRPO group-relative advantage handles credit assignment).

    TODO(conv-point): replace the stub body with the real Council-driven
    rollout once Session 13 ships.
    """
    rewards = []
    for completion, t, s in zip(completions, task, seed):
        # TODO(conv-point):
        # try:
        #     routing_action = RoutingAction.model_validate_json(completion)
        # except ValidationError:
        #     rewards.append(-1.0)  # invalid JSON → terminal-equivalent penalty
        #     continue
        # council = Council(routing_policy=trainable_router_from(routing_action),
        #                   env=make_env(), brains=cortex_brains)
        # episode_return = council.run_episode(task=t, seed=int(s)).total_reward
        # rewards.append(episode_return)
        rewards.append(0.0)  # stub
    return rewards

## 9. GRPO training

Same TRL `GRPOTrainer` shape as the B1 notebook. ~300 steps × group size 4 = ~1200 router decisions × full-episode rollouts. Wall-clock ~1.5 hours on a100-large at convergence point (dominated by brain-LLM rollouts, not router fine-tuning).

In [ ]:
from trl import GRPOConfig, GRPOTrainer

MAX_TRAIN_STEPS = 300
GROUP_SIZE = 4
MAX_PROMPT_LEN = 512
MAX_COMPLETION_LEN = 256  # router output is structured JSON (M-FR-11)

training_args = GRPOConfig(
    output_dir="/content/cortex_router_grpo_output",
    learning_rate=5e-6,
    per_device_train_batch_size=GROUP_SIZE,
    gradient_accumulation_steps=1,
    num_generations=GROUP_SIZE,
    max_prompt_length=MAX_PROMPT_LEN,
    max_completion_length=MAX_COMPLETION_LEN,
    max_steps=MAX_TRAIN_STEPS,
    save_steps=100,
    logging_steps=5,
    report_to="none",
    bf16=True,
    optim="adamw_8bit",
    temperature=0.8,
    use_vllm=True,
    vllm_mode="colocate",
    seed=42,
)

trainer = GRPOTrainer(
    model=router_model,
    processing_class=router_tokenizer,
    reward_funcs=[cortex_router_reward],
    args=training_args,
    train_dataset=train_dataset,
)
print("Router GRPOTrainer constructed.")
print("# TODO(conv-point): uncomment trainer.train() once Cortex Session 13 lands.")
# trainer.train()

## 10. Save the trained router LoRA to HF Hub

In [ ]:
from huggingface_hub import HfApi

HUB_REPO = "Angshuman28/crisisworld-cortex-router-llm"

router_model.save_pretrained("/content/cortex_router_lora")
router_tokenizer.save_pretrained("/content/cortex_router_lora")

api = HfApi()
api.create_repo(HUB_REPO, exist_ok=True, repo_type="model", private=False, token=HF_TOKEN)
api.upload_folder(
    folder_path="/content/cortex_router_lora",
    repo_id=HUB_REPO,
    repo_type="model",
    token=HF_TOKEN,
)
print(f"Saved to https://huggingface.co/{HUB_REPO}")

## 11. Eval: B6 (trained LLM router) vs B3 (deterministic) on 3 tasks

**TODO(conv-point):** the comparison loop below requires both B3CortexFixedRouter (deterministic) and a way to swap the trainable router into Council.routing_policy. The eval is the headline result for the convergence point — its sign decides whether B6 ships or B3 ships per the Phase 7 hard-exit gate.

Decision rule (per spec): if reward over training steps is INCREASING, ship B6. Else ship B3.

In [ ]:
# TODO(conv-point):
# from cortex.routing_policy import DeterministicRouter, TrainableRouter
# from cortex.council import Council
# from baselines.cortex_fixed_router import B3CortexFixedRouter
#
# def run_b6_episode(task, seed):
#     trainable = TrainableRouter(
#         model=router_model,
#         tokenizer=router_tokenizer,
#         system_prompt=ROUTER_SYSTEM_PROMPT,
#         featurize=metacog_state_to_prompt,
#     )
#     council = Council(routing_policy=trainable, env=make_env(), brains=cortex_brains)
#     return council.run_episode(task=task, seed=seed).total_reward
#
# def run_b3_episode(task, seed):
#     b3 = B3CortexFixedRouter(env=make_env())
#     return b3.run_episode(task=task, seed=seed).total_reward
#
# b6_results = {t: run_b6_episode(t, 0) for t in TASKS}
# b3_results = {t: run_b3_episode(t, 0) for t in TASKS}
# print(f"B6 (trained LLM router): {b6_results}")
# print(f"B3 (deterministic): {b3_results}")

print("Eval cell skeleton — uncomment when Cortex Session 13 ships")

## 12. Plot training reward curve

Phase 7 hard-exit gate watches this curve. If reward is increasing across training steps → ship B6. Else → ship B3.

In [ ]:
import matplotlib.pyplot as plt

log_path = "/content/cortex_router_grpo_output/trainer_state.json"
if os.path.exists(log_path):
    import json as _json

    with open(log_path) as fh:
        state = _json.load(fh)
    history = state.get("log_history", [])
    rewards = [entry["reward"] for entry in history if "reward" in entry]
    if rewards:
        fig, ax = plt.subplots(figsize=(8, 4))
        ax.plot(rewards)
        ax.set_xlabel("GRPO step")
        ax.set_ylabel("Mean reward")
        ax.set_title("Cortex small-LLM router — training reward")
        plt.tight_layout()
        plt.show()
    else:
        print("No reward entries yet — uncomment trainer.train() at conv-point")
else:
    print("No trainer state yet — uncomment trainer.train() at conv-point")